In [53]:
using LinearAlgebra
using JLD2
using Plots

In [54]:

args=[0.36,-0.6,-0.6,0.0,0.0,20.8,107.7,20.8,-107.7,6.0,70.0,3.89,-23.8,1.0,3.0,3.0]
parameters=args[1:13]
holenum=Int(args[14])
trytime=Int(args[15])
Nq=Int(args[16]);


    mt=parameters[1]
    mm=parameters[2]
    mb=parameters[3]
    Vt=parameters[4]
    ϕt=parameters[5]/180*π
    Vm=parameters[6]
    ϕm=parameters[7]/180*π
    Vb=parameters[8]
    ϕb=parameters[9]/180*π
    ϵr=parameters[10]
    Eg=parameters[11]
    θ=parameters[12]/180*π
    w=parameters[13]
  
  
  
  
  
  
  
    alattice=0.352# In units of NM
    blattice=4*π/(√3*alattice)
  
    
    
    bm=blattice*θ
    am=4*π/(√3*bm);
    constt=38.09981949*1/(mt)
    constm=38.09981949*1/(mm)
    constb=38.09981949*1/(mb) # the 38 is \hbar^2/(2me*nm^2) in units of meV
    constq=1/(Nq^2*3^(1/2)/2*am^2)*1/ϵr*9047.5636
  
    
    b1=bm*[1,0]
    b2=bm*[-1/2,√3/2]
    
  
    a1m=am*[√3/2,1/2]
    a2m=am*[0,1]
  
    
    T1=b1/(Nq)
    T2=b2/(Nq)
    
    κp=bm*[-1/2,1/(2√3)]
    κm=bm*[-1/2,-1/(2√3)]
    
    
    b1T=Int.(round.(inv([T1 T2])*b1))
    b2T=Int.(round.(inv([T1 T2])*b2))
    
    
    
    allowedq=Vector{Int64}[]
    for ja in 0:Nq-1,jb in 0:Nq-1
        push!(allowedq,[ja,jb])
    end
  
    
    
    
    wave=Vector{Int64}[]
    cutoff=18
    cutoffstandard=3.01*bm
    for ja in -cutoff:cutoff, jb in -cutoff:cutoff
        gtest=ja*b1+jb*b2;
        if (gtest[1]^2+gtest[2]^2)<cutoffstandard^2
            push!(wave,ja*b1T+jb*b2T)
        end
    end
    dimension=3*length(wave)
   
    shift=1/2*T1+1/2*T2
  
  
  
  
    single_Ham=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    single_MoirePo=zeros(ComplexF64,dimension,dimension)
    tunnel=zeros(ComplexF64,dimension,dimension)
    M_tunnel=zeros(ComplexF64,dimension,dimension)
    seed_tunnel=zeros(ComplexF64,dimension,dimension)
    
    single_eigenvalue=[[zeros(Float64,dimension) for _ in 1:2] for _ in 1:Nq^2]
    single_eigenvector=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    uncoupled_eigenvector=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    seed_eigenvector=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
   
    
    for jc in eachindex(wave)
      pos=findfirst(item->item==wave[jc]-b1T,wave)
      if pos≠nothing
        single_MoirePo[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]=diagm([Vt*exp(im*ϕt),Vm*exp(im*ϕm),Vb*exp(im*ϕb)])
      end
      
    
      pos=findfirst(item->item==wave[jc]+b2T+b1T,wave)
      if pos≠nothing
        single_MoirePo[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]=diagm([Vt*exp(im*ϕt),Vm*exp(im*ϕm),Vb*exp(im*ϕb)])
      end
    
      pos=findfirst(item->item==wave[jc]-b2T,wave)
      if pos≠nothing
        single_MoirePo[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]=diagm([Vt*exp(im*ϕt),Vm*exp(im*ϕm),Vb*exp(im*ϕb)])
      end
    end
    
    
    single_MoirePo=single_MoirePo+single_MoirePo'
    
    for jc in eachindex(wave)
     
      tunnel[3*(jc-1)+3,3*(jc-1)+2]=w
    
      pos=findfirst(item->item==wave[jc]+b2T+b1T,wave)
      if pos≠nothing
     
        tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    
      pos=findfirst(item->item==wave[jc]+b2T,wave)
      if pos≠nothing
       
        tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    end
    
    tunnel=tunnel+tunnel'
    
    for jc in eachindex(wave)
  
      M_tunnel[3*(jc-1)+3,3*(jc-1)+2]=w
    
      pos=findfirst(item->item==wave[jc]-b2T-b1T,wave)
      if pos≠nothing
       
        M_tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    
      pos=findfirst(item->item==wave[jc]-b2T,wave)
      if pos≠nothing
   
        M_tunnel[3*(jc-1)+3,3*(pos-1)+2]=w
      end
    end
    
    M_tunnel=M_tunnel+M_tunnel'
  
    for jc in eachindex(wave)
     
      seed_tunnel[3*(jc-1)+1,3*(jc-1)+2]+=2.0
      seed_tunnel[3*(jc-1)+1,3*(jc-1)+3]+=2.0
    end
  
    for jc in eachindex(wave)
      pos=findfirst(item->item==wave[jc]-b1T,wave)
      if pos≠nothing
        seed_tunnel[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]+=diagm([1*exp(im*ϕt),1*exp(im*ϕm),1*exp(im*ϕb)])
      end
      
    
      pos=findfirst(item->item==wave[jc]+b2T+b1T,wave)
      if pos≠nothing
        seed_tunnel[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]+=diagm([1*exp(im*ϕt),1*exp(im*ϕm),1*exp(im*ϕb)])
      end
    
      pos=findfirst(item->item==wave[jc]-b2T,wave)
      if pos≠nothing
        seed_tunnel[3*(jc-1)+1:3*jc,3*(pos-1)+1:3*pos]+=diagm([1*exp(im*ϕt),1*exp(im*ϕm),1*exp(im*ϕb)])
      end
    end
    
    seed_tunnel=seed_tunnel+seed_tunnel'
    
    
    
    
    Threads.@threads for ja in 1:Nq^2
      
      
      k=allowedq[ja][1]*T1+allowedq[ja][2]*T2
    
      for jb in eachindex(wave)
       single_Ham[ja][1][3*(jb-1)+1,3*(jb-1)+1]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2-κp+shift)^2*constt+Eg #This is the configuration
       single_Ham[ja][1][3*(jb-1)+2,3*(jb-1)+2]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2-κp+shift)^2*constm
       single_Ham[ja][1][3*(jb-1)+3,3*(jb-1)+3]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2-κm+shift)^2*constb
      end
    
      for jb in eachindex(wave)
        single_Ham[ja][2][3*(jb-1)+1,3*(jb-1)+1]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2+κp+shift)^2*constt+Eg
        single_Ham[ja][2][3*(jb-1)+2,3*(jb-1)+2]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2+κp+shift)^2*constm
        single_Ham[ja][2][3*(jb-1)+3,3*(jb-1)+3]=norm(k+wave[jb][1]*T1+wave[jb][2]*T2+κm+shift)^2*constb
       end
      
    
       
      FFF=eigen(single_Ham[ja][1])
       uncoupled_eigenvector[ja][1]=FFF.vectors
  
      FFF=eigen(single_Ham[ja][2])
       uncoupled_eigenvector[ja][2]=FFF.vectors
      
       FFF=eigen(single_Ham[ja][1]+seed_tunnel)
       seed_eigenvector[ja][1]=FFF.vectors
  
      FFF=eigen(single_Ham[ja][2]+seed_tunnel)
       seed_eigenvector[ja][2]=FFF.vectors
      
    
    
      single_Ham[ja][1]+=single_MoirePo+tunnel
      single_Ham[ja][2]+=single_MoirePo+M_tunnel
     
      FFF=eigen(single_Ham[ja][1])
      single_eigenvalue[ja][1]=real(FFF.values)
      single_eigenvector[ja][1]=FFF.vectors
    
      FFF=eigen(single_Ham[ja][2])
      single_eigenvalue[ja][2]=real(FFF.values)
      single_eigenvector[ja][2]=FFF.vectors
  
    end
    
  
  
  
  
     
    BG_DensityMatrix=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    for ja in 1:Nq^2, vi in 1:2, jb in 1:length(wave)*2
      BG_DensityMatrix[ja][vi]+=(uncoupled_eigenvector[ja][vi][:,jb]*(uncoupled_eigenvector[ja][vi][:,jb])') 
      
    end
     input_DensityMatrix=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]
    
    for ja in 1:Nq^2, vi in 1:1
      A=10^(-1)*randn(ComplexF64,dimension,dimension)
        input_DensityMatrix[ja][vi]=A+A'
    end
  
    for ja in 1:Nq^2, vi in 2:2
      A=10^(-1)*randn(ComplexF64,dimension,dimension)
        input_DensityMatrix[ja][vi]=A+A'
    end
    
  
  
  
  
    for ja in 1:Nq^2, vi in 1:1, jb in 1:length(wave)*2
     input_DensityMatrix[ja][vi]+=(seed_eigenvector[ja][vi][:,jb]*(seed_eigenvector[ja][vi][:,jb])') 
    end
    for ja in 1:Nq^2, vi in 2:2, jb in 1:length(wave)*2-2
      input_DensityMatrix[ja][vi]+=(seed_eigenvector[ja][vi][:,jb]*(seed_eigenvector[ja][vi][:,jb])') 
    end
    for ja in 1:Nq^2, vi in 2:2, jb in length(wave)*2+1:length(wave)*2+1
      input_DensityMatrix[ja][vi]+=(seed_eigenvector[ja][vi][:,jb]*(seed_eigenvector[ja][vi][:,jb])') 
    end
  
    input_DensityMatrix-=BG_DensityMatrix
   
   


9-element Vector{Vector{Matrix{ComplexF64}}}:
 [[0.04458013451848337 + 0.0im -0.1646197653363856 - 0.09243810805045358im … -0.09158219724766999 + 0.13293038438775368im 0.049942299223130905 - 0.15017477427424208im; -0.1646197653363856 + 0.09243810805045358im -0.10217896207610577 + 0.0im … 0.12225997829037731 - 0.14389607891013456im 0.031079840781069134 + 0.11853218253888484im; … ; -0.09158219724766999 - 0.13293038438775368im 0.12225997829037731 + 0.14389607891013456im … -0.11936993234808779 + 0.0im -0.10677483191306612 + 0.09936687040553693im; 0.049942299223130905 + 0.15017477427424208im 0.031079840781069134 - 0.11853218253888484im … -0.10677483191306612 - 0.09936687040553693im 0.08865399948996511 + 0.0im], [-0.10997753015504408 + 0.0im 0.09146897865737932 - 0.040208200813539006im … 0.16962498132698683 - 0.07125563194698836im -0.041405445019898096 + 0.020111985391108384im; 0.09146897865737932 + 0.040208200813539006im 0.1332364321365329 + 0.0im … 0.05267005317250296 - 0.08853419995434898

In [56]:
pp=load("data_output/0.36mt-0.6mm-0.6mb0.0Vt0.0phit20.8Vm107.7phim20.8Vb-107.7phib6.0er70.0Eg3.89theta-23.8w1.0holenum3.0trytime3.0Nq.jld2")
   

Dict{String, Any} with 10 entries:
  "HFdensity"                => [0.666447+1.07295e-16im 0.69742-9.82287e-17im ……
  "parameters"               => [0.36, -0.6, -0.6, 0.0, 0.0, 20.8, 107.7, 20.8,…
  "single_eigenvalue"        => [[[-1686.8, -1640.81, -1588.06, -1480.25, -1430…
  "single_chern"             => ComplexF64[-1.00001+1.76697e-17im, 1.0+3.09221e…
  "energy"                   => -195.24
  "HF_eigenvalue"            => [[[-1748.04, -1702.14, -1650.28, -1542.19, -149…
  "bound"                    => -17.1287
  "HF_eigenvector"           => Vector{Matrix{ComplexF64}}[[[-1.25922e-11+1.565…
  "DIIS_input_DensityMatrix" => Vector{Vector{Matrix{ComplexF64}}}[[[[2.19246e-…
  "HF_chern"                 => ComplexF64[0.999972+9.94117e-17im, 0.999999-8.5…

In [57]:
pp["HF_eigenvector"]

9-element Vector{Vector{Matrix{ComplexF64}}}:
 [[-1.2592194464071472e-11 + 1.5656227635350598e-12im -3.037543447793335e-10 - 3.262420203019398e-11im … 2.3714387522639328e-6 - 4.614873018448832e-5im -1.1912461045508844e-8 + 3.7287552290485687e-8im; 7.307075936594385e-10 - 2.457601195520765e-10im 6.88938456180466e-8 - 4.3106584124679746e-8im … -5.496573679025796e-9 - 2.619149317074502e-10im -1.3631771449501991e-11 + 6.815473232147102e-12im; … ; 0.06176981919588065 - 0.0029340463591669055im 1.4126617899420588e-5 + 0.0003541939118116644im … 9.449778786765981e-12 + 1.3534437672381058e-11im -1.2347288776301763e-8 + 1.3606619364810422e-8im; 0.9937010064991496 + 0.0im 0.0032870197865379498 - 0.0im … -5.281292746784751e-10 - 0.0im 4.633892731351634e-7 - 0.0im], [-8.064211620971246e-11 - 9.597333829179904e-10im 4.5416135775190425e-7 - 7.41788197330611e-9im … -4.459445258726024e-5 - 2.0169067401135525e-5im -2.359174448669548e-8 - 3.639629243828841e-8im; 3.4997965458702e-8 - 2.4704512908865416e-9i

In [65]:
trial_densitymatrix=[[zeros(ComplexF64,dimension,dimension) for _ in 1:2] for _ in 1:Nq^2]


for ja in 1:Nq^2, vi in 2:2, jb in length(wave)*2:length(wave)*2
    trial_densitymatrix[ja][vi]+=(pp["HF_eigenvector"][ja][vi][:,jb]*(pp["HF_eigenvector"][ja][vi][:,jb])') 
end

In [66]:
function Densitymap(a1m::Vector{Float64},a2m::Vector{Float64},wave::Vector{Vector{Int}},input_DensityMatrix::Vector{Vector{Matrix{ComplexF64}}})::Tuple{Array{ComplexF64},Array{ComplexF64},Array{ComplexF64}}
    dimension=3*length(wave)  
       N3=50
      xgrid=zeros(Float64,N3,N3)
      ygrid=zeros(Float64,N3,N3)
      zgrid=zeros(ComplexF64,N3,N3,2,3)
      Total_Density=[zeros(ComplexF64,dimension,dimension) for _ in 1:2]
      for jk1 in 1:Nq^2
         Total_Density+=input_DensityMatrix[jk1]
      end
      
      for ja in 1:N3, jb in 1:N3
        xgrid[ja,jb]=(ja/N3*a1m[1]+jb/N3*a2m[1])
        ygrid[ja,jb]=(ja/N3*a1m[2]+jb/N3*a2m[2])
        rvec=ja/N3*a1m+jb/N3*a2m
        for vi in 1:2, Li in 1:3, jc in eachindex(wave), jd in eachindex(wave)
          gvec=[T1 T2]*(wave[jc]-wave[jd])
         zgrid[ja,jb,vi,Li]+=Total_Density[vi][3*(jc-1)+Li,3*(jd-1)+Li]*exp(im*(gvec[1]*rvec[1]+gvec[2]*rvec[2]))
        end
      
      end
  
      return xgrid,ygrid,zgrid
  end

Densitymap (generic function with 1 method)

In [67]:
xgrid,ygird,HFdensity=Densitymap(a1m,a2m,wave,trial_densitymatrix)

(ComplexF64[0.08979999695096334 + 0.0im 0.08979999695096334 + 0.0im … 0.08979999695096334 + 0.0im 0.08979999695096334 + 0.0im; 0.17959999390192669 + 0.0im 0.17959999390192669 + 0.0im … 0.17959999390192669 + 0.0im 0.17959999390192669 + 0.0im; … ; 4.400199850597203 + 0.0im 4.400199850597203 + 0.0im … 4.400199850597203 + 0.0im 4.400199850597203 + 0.0im; 4.489999847548167 + 0.0im 4.489999847548167 + 0.0im … 4.489999847548167 + 0.0im 4.489999847548167 + 0.0im], ComplexF64[0.15553815723859876 + 0.0im 0.2592302620643313 + 0.0im … 5.132759188873759 + 0.0im 5.236451293699492 + 0.0im; 0.20738420965146503 + 0.0im 0.3110763144771975 + 0.0im … 5.184605241286626 + 0.0im 5.288297346112358 + 0.0im; … ; 2.644148673056179 + 0.0im 2.7478407778819114 + 0.0im … 7.62136970469134 + 0.0im 7.725061809517072 + 0.0im; 2.6959947254690455 + 0.0im 2.7996868302947777 + 0.0im … 7.673215757104206 + 0.0im 7.7769078619299385 + 0.0im], [0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.

In [72]:
(sum(HFdensity[:,:,2,3]))/50^2*1/9

0.0662857931772934 + 1.8526872950572254e-19im